## This is the code to train the model and acquire influence for Number of Samples Experiment

**Default Code**:   
The current default code is a runnable sample. It runs on the synthetic dataset generated with sklearn's make_classification function. The default version code provides the synthetic dataset with 16500 samples and 160 features in total. The separation is set to 5 to make sure the dataset is distinguishable by the model. All features are set to be informative to ensure they are of equal importance. The dataset has only two labels, so it is a binary classification problem. The default setting will then generate the training set and test set from the pool. The default training sample size is 8000, and the test size is 500. The number of features is set to 10. The model in default will be a Simple FeedForward Neural Network constructed by TensorFlow. The Influence Estimation methods we provide by default are the Influence Function and TracIn. If you simply press 'play', the default code will generate ranked influence lists for both Influence Function and TracIn with respect to the above mentioned setting in the root directory. The result lists could then be fed into other analyses.

**By default, this is exactly the same code as the base code. Please refer to the base code for more detailed explanation.**

**Guideline**:  
Read in / Construct Datasets -> **Choose the Training Sample Size** -> Pre-processing -> Model Training -> Influence Estimation -> Store the Ranked Influence lists -> **Change the Training Sample Size and Repeat all the process** -> ... -> **After all the training and estimation, feed the results into the analysis code**  (Remember to change the file name in the last block to save lists in different settings.)

# Import Area

Here is the area to place all the import codes. You don't need to change here unless you want to customise in later sections.

In [32]:
import tensorflow as tf
import keras
from keras.utils import to_categorical
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [33]:
from keras import Sequential
from keras.layers import Dense, BatchNormalization, Dropout
from keras.losses import CategoricalCrossentropy
from keras.optimizers import Adam

In [34]:
from deel.influenciae.common import InfluenceModel, ExactIHVP
from deel.influenciae.influence import FirstOrderInfluenceCalculator
from deel.influenciae.utils import ORDER
from deel.influenciae.trac_in import TracIn

In [35]:
import random
from keras.optimizers import SGD

In [36]:
from sklearn.datasets import make_classification

# Dataset Construction Area

**You can use any dataset you wish here, either regression or classification. But in default, since we are using influenciae's IF and TC method, make sure they are split into train and test sets, and then stored as tensorflow dataset format. If you only want to change the dataset, you can only change the code in the first two blocks to read in/ generate your own dataset. But remember to have features X and target y before going to the third block. Also, if you wish to everything on your own, remember to add id inside the dataset.** Since our default code is for classification, the regression might need a lot of changes in all the following sections.


**Input**: Dataset chosen(Usually in Features X and Target y format)  
**Output**: Tensorflow format Train and Test Set  
**Guideline**: Input -> Turn into Dataframe and add ID -> Pre-Processing -> Change the format to Tensorflow -> Output

The default code now produces a synthetic dataset with 16500 pool, 160 features with binary classification problems. The later options will turn that into a 10 features, 8000 train set and 500 test set sample. Both sets will then be turned into TensorFlow format and will wait for training.

**The most important thing in this code is changing the train size. In this experiment, all the other things are fixed, but the number of training sample is changing to test on different number of samples.**

1. Set your default setting here. train_pool + test_size = Total Dataset Size. train_sizes determines the later subset data. Sep to make sure the dataset is distinguishable.

In [37]:
train_pool = 16000
test_size = 500
n_features=10
seed=42
ratios = [(9,1), (8,2), (7,3), (6,4), (5,5)]
sep = 1.5

2. Construct the Synthetic Dataset with Make Classification here. **Could replace this with other datasets with X and y.**

In [38]:
total_samples = train_pool + test_size

X, y = make_classification(n_samples=total_samples,
                           n_features=n_features,
                           n_informative=n_features,
                           n_redundant=0,
                           n_repeated=0,
                           n_classes=2,
                           class_sep=sep,
                           random_state=seed)

3. Turn the X and y into dataframe for easy further processing. Add ID column to easy retrieve samples within IF/TC.

In [39]:
df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(n_features)])
df['label'] = y
df['id'] = np.arange(1, len(df) + 1)
print(df)

       feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0      -2.042923   2.447225   1.459419  -4.895205  -1.931741   2.439100   
1      -3.021719  -2.189454   1.594643  -0.936749   6.580163  -0.856511   
2       1.744477  -1.747769  -1.803517  -3.028068  -0.517037   3.262006   
3      -0.372644   3.341254   1.849793   0.905480   0.359212   1.766193   
4      -1.972171  -3.053477   1.529830   1.741359   0.858712   1.675452   
...          ...        ...        ...        ...        ...        ...   
16495  -1.318844  -1.342788   0.624967  -0.463707  -0.735329   1.971186   
16496  -3.063709  -0.202935  -0.534888  -6.109804  -0.977436   1.353489   
16497   0.408280  -2.252040   2.995770  -0.442872  -0.904981   2.307891   
16498  -0.117513   3.516412  -3.900624  -2.741435  -4.439399  -0.522876   
16499   3.901795  -1.910619  -0.779033  -0.536239  -1.471359   0.685229   

       feature_7  feature_8  feature_9  feature_10  label     id  
0      -0.694749  -3.070256  -3.

In [40]:
exact_size = 8500

In [41]:
cur_ratio = ratios[2]
print(cur_ratio)

(7, 3)


In [42]:
major, minor = cur_ratio

In [43]:
df0 = df[df.label == 0]  
df1 = df[df.label == 1] 

In [44]:
# t0 = int(exact_size * major / (major + minor))
# t1 = exact_size - t0 
# print(t0,t1)

In [45]:
t0 = 5850
t1 = 2650

In [46]:
s0 = df0.sample(n=t0, random_state=seed)
s1 = df1.sample(n=t1, random_state=seed)

In [47]:
df = pd.concat([s0, s1], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)

In [48]:
print(df)
print(df["label"].value_counts())

      feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0      2.000551   6.107697   0.922888  -0.645629  -2.044902   6.067214   
1     -1.482890   3.501781   1.723413  -0.987130   4.693062   3.417544   
2     -1.461155   4.709173   1.276742   2.226696   1.542537   5.980747   
3      2.349369   2.618102   0.891645  -1.675392   0.253586  -0.248629   
4     -3.087668   2.723457   0.627227   3.464238  -0.751775   0.861041   
...         ...        ...        ...        ...        ...        ...   
8495  -0.392552  -4.158898   4.090073   1.949615  -0.431500  -0.218027   
8496   3.581924  -1.483711  -0.054275  -2.953603  -1.952614  -2.313367   
8497   2.125435  -1.880934  -2.997253  -3.589638  -2.528225   0.119474   
8498  -1.806808  -0.704839   1.698431  -3.602001   0.844647   1.772143   
8499  -1.416175  -0.981882  -0.853378  -0.154625  -5.670463  -1.923880   

      feature_7  feature_8  feature_9  feature_10  label     id  
0     -1.856265  -2.283272  -2.149731   -5.91

In [49]:
# id_label_df = df[["id", "label"]].copy()
# print(id_label_df)
# id_label_df.to_csv("9_1_class_labelIDs.csv",index = False)

In [50]:
n0 = test_size //2
n1 = test_size - n0

In [51]:
g = df.groupby("label", group_keys=False)
test_df = pd.concat([
    g.get_group(0).sample(n=n0, random_state=42, replace=False),
    g.get_group(1).sample(n=n1, random_state=42, replace=False),
]).sample(frac=1, random_state=42)

train_df = df.drop(test_df.index).reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

In [52]:
print(test_df.groupby("label").get_group(0))
print(test_df["label"].value_counts())

     feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
1     0.834430   0.463094   1.273167  -1.544390   0.188683  -1.068960   
3     0.607505   1.762207  -0.338045  -1.004661  -1.166495  -0.464131   
4     1.942862   0.143290  -0.346069   3.511778   2.656194  -3.307865   
7     6.311960   1.746709   1.541404   1.978819  -1.092200  -2.673320   
8     0.916350  -0.132366  -2.954575  -2.036922  -0.780542   2.473928   
..         ...        ...        ...        ...        ...        ...   
492  -2.945985   1.151929   2.848188  -4.928831   1.101044   4.751279   
493  -1.706920   0.867694  -0.438359  -0.844137   1.334499  -2.068917   
494   1.130985   3.047467  -0.718695   0.831319  -0.792173  -2.747026   
495   1.451834   1.841968  -5.509639  -1.650674  -1.608023  -0.265070   
499   0.139369  -0.459396   0.936389  -4.317837  -4.288256  -2.819963   

     feature_7  feature_8  feature_9  feature_10  label     id  
1    -0.365911  -0.208600   2.786547   -1.965793      0   

In [53]:
print(train_df["label"].value_counts())

label
0    5600
1    2400
Name: count, dtype: int64


4. Here, we choose the subset of the full dataset. By setting features_to_test, we have the subset feature size. By changing nested_train_dfs, we have different sample sizes.

**The most important thing in this experiment is the following code**:  
Based on the train_sizes=[1000,2000,3000,4000,5000,6000,7000,8000,9000,10000] defined above, we could map the number of training samples with the following code. By choosing the number in [], we could modify the train set size. Therefore, only changing the following code block is enough to produce the experiment result successfully.

In [54]:
train_df["clean_label"] = train_df["label"].copy()

noise_ratio = 0.20          # Total desired noise
noise_seed = 42

rng = np.random.default_rng(noise_seed)

# Get indices of each class
label0_idx = train_df.index[train_df["label"] == 0].to_numpy()
label1_idx = train_df.index[train_df["label"] == 1].to_numpy()

# Flip 10% from each class
flip_ratio_per_class = noise_ratio

n_flip_0 = int(len(label0_idx) * flip_ratio_per_class)
n_flip_1 = int(len(label1_idx) * flip_ratio_per_class)

flip0 = rng.choice(label0_idx, size=n_flip_0, replace=False)
flip1 = rng.choice(label1_idx, size=n_flip_1, replace=False)

# Combine flipped indices
noisy_indices = np.concatenate([flip0, flip1])

# Mark noisy samples
train_df["is_noisy"] = 0
train_df.loc[noisy_indices, "is_noisy"] = 1

# Flip labels
train_df.loc[noisy_indices, "label"] = (
    1 - train_df.loc[noisy_indices, "label"]
)

train_df["noisy_label"] = train_df["label"]

print("Label 0 flipped:", n_flip_0)
print("Label 1 flipped:", n_flip_1)
print("Total flipped:", train_df["is_noisy"].sum())

print(
    train_df.groupby("clean_label")["is_noisy"]
            .agg(["sum", "count", "mean"])
)

Label 0 flipped: 1120
Label 1 flipped: 480
Total flipped: 1600
              sum  count  mean
clean_label                   
0            1120   5600   0.2
1             480   2400   0.2


In [55]:
# Feature columns used by the model
selected_features = [
    col for col in train_df.columns
    if col.startswith("feature_")
]

# Preserve IDs separately
train_ids_original = train_df["id"].to_numpy()

# Scale IDs only for the influence pipeline
IDs = (
    train_ids_original
    .reshape(-1, 1)
    .astype(np.float32)
    / 1e10
)

# Model features
X_train_features = train_df[
    selected_features
].to_numpy(dtype=np.float32)

# Append the ID column, as required by your existing influence code
X_train = np.hstack([
    X_train_features,
    IDs
])

# Use the corrupted labels for training
y_train_1d = train_df[
    "noisy_label"
].to_numpy(dtype=np.int64)

y_train = to_categorical(
    y_train_1d,
    num_classes=2
)

print(X_train.shape)
print(y_train.shape)

(8000, 11)
(8000, 2)


In [56]:
# X_train = train_df.drop(columns=["label"])
# y_train = train_df["label"]
# IDs = X_train["id"].values.reshape(-1, 1).astype(np.float32)
# IDs = IDs  / 1e10

# X_train = X_train.drop(columns=["id"]).values.astype(np.float32)
# X_train = np.hstack((X_train, IDs))
# y_train = to_categorical(y_train.values,num_classes=2)

# print(X_train)

In [57]:
X_test = test_df.drop(columns=["label"])
y_test = test_df["label"]
IDs = X_test["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_test = X_test.drop(columns=["id"]).values.astype(np.float32)
X_test = np.hstack((X_test, IDs))
y_test = to_categorical(y_test.values,num_classes=2)

print(X_test.shape)
print(y_test.shape)

(500, 11)
(500, 2)


5 (Optional) The following code below can display the samples distribution. Uncomment them to acquire the distribution plot.

In [58]:
# X_all = np.vstack([X_train, X_test])

# y_train_1d = np.argmax(y_train, axis=1)
# y_test_1d  = np.argmax(y_test, axis=1)
# y_all_1d = np.hstack([y_train_1d, y_test_1d])

In [59]:
# from sklearn.metrics import pairwise_distances
# from sklearn.manifold import MDS
# import seaborn as sns
# import matplotlib.pyplot as plt

In [60]:
# D = pairwise_distances(X_all) 

In [61]:
# X_mds = MDS(n_components=2, dissimilarity='precomputed', random_state=0).fit_transform(D)

In [62]:
# sns.scatterplot(x=X_mds[:,0], y=X_mds[:,1], hue=y_all_1d)
# plt.title("MDS – preserves original distances")

6. Now we have the train_ds and test_ds for training

In [63]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

# Training Area

**Could modify the model as you wish here. Again, in default, influenciae relies on TensorFlow, so use the TensorFlow model if you only want to change the model. Remember: Store the InfluenceModel into the model_list with the loss function. The InfluenceModel will be used to obtain influence later. If you don't change the estimation methods, then the final output at this step shall always be the model_list**

**Input**:Train and Test Set from Data Construction Section   
**Output**: Model List  
**Guideline**: Input -> Define the Model and Hyperparameters -> Train the Model -> Output

Always remember to train the model, get the influence model and store that in model list, unless you wish to change the estimation methods.

The default code now use the train and test set generated from the last section to train the model. The default hyperparameters are: 300 Epochs, Simple FeedForward Neural Network, CategoricalCrossEntropy Loss function, SGD optimizer. Within each epoch, the current model will be turned into an Influence Model and stored inside a model list. After the training, the model list will be passed to next section for influence estimation.

In [64]:
from tensorflow.keras.regularizers import l2

1. **Could modify the model as you wish here as long as it is tensorflow.** Just remember: Store the InfluenceModel into the model_list with the loss function

In [65]:
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)),  
    BatchNormalization(momentum=0.9),
    Dropout(0.0),
    Dense(8, activation='relu'),
    Dense(y_train.shape[1])
])
loss_fn = CategoricalCrossentropy(from_logits=True)
optimizer = SGD(learning_rate=0.001, momentum=0.9)
model.compile(loss=loss_fn, optimizer=optimizer, metrics=['accuracy'])

epochs = 300
unreduced_loss_fn = CategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
model_list = []
initial_model = tf.keras.models.clone_model(model)
initial_model.set_weights(model.get_weights())
model_list.append(InfluenceModel(initial_model, start_layer=-1, loss_function=unreduced_loss_fn))
for i in range(epochs):
  model.fit(train_ds.batch(256), epochs=1, validation_data=test_ds.batch(256), verbose=2)
  checkpoint_model = tf.keras.models.clone_model(model)
  checkpoint_model.set_weights(model.get_weights())
  model_list.append(InfluenceModel(checkpoint_model, start_layer=-1, loss_function=unreduced_loss_fn))
base_loss, acc = model.evaluate(test_ds.batch(32), verbose=2)
print(base_loss)

32/32 - 1s - loss: 0.8299 - accuracy: 0.5829 - val_loss: 0.8985 - val_accuracy: 0.4620 - 848ms/epoch - 26ms/step
32/32 - 0s - loss: 0.7218 - accuracy: 0.5850 - val_loss: 0.7254 - val_accuracy: 0.5440 - 65ms/epoch - 2ms/step
32/32 - 0s - loss: 0.6750 - accuracy: 0.6093 - val_loss: 0.6482 - val_accuracy: 0.6160 - 64ms/epoch - 2ms/step
32/32 - 0s - loss: 0.6527 - accuracy: 0.6386 - val_loss: 0.6054 - val_accuracy: 0.6760 - 62ms/epoch - 2ms/step
32/32 - 0s - loss: 0.6387 - accuracy: 0.6616 - val_loss: 0.5774 - val_accuracy: 0.7260 - 63ms/epoch - 2ms/step
32/32 - 0s - loss: 0.6286 - accuracy: 0.6771 - val_loss: 0.5571 - val_accuracy: 0.7500 - 60ms/epoch - 2ms/step
32/32 - 0s - loss: 0.6207 - accuracy: 0.6898 - val_loss: 0.5412 - val_accuracy: 0.7760 - 58ms/epoch - 2ms/step
32/32 - 0s - loss: 0.6144 - accuracy: 0.6942 - val_loss: 0.5281 - val_accuracy: 0.7860 - 88ms/epoch - 3ms/step
32/32 - 0s - loss: 0.6092 - accuracy: 0.6982 - val_loss: 0.5171 - val_accuracy: 0.7820 - 63ms/epoch - 2ms/step

In [66]:
train_logits = model.predict(
    X_train,
    batch_size=256,
    verbose=0
)

# Calculate one loss value per sample using the corrupted labels
per_sample_loss_fn = CategoricalCrossentropy(
    from_logits=True,
    reduction=tf.keras.losses.Reduction.NONE
)

training_losses = per_sample_loss_fn(
    y_train,
    train_logits
).numpy()

print(training_losses.shape)
print(pd.Series(training_losses).describe())

(8000,)
count    8000.000000
mean        0.525866
std         0.530716
min         0.023511
25%         0.199609
50%         0.277741
75%         0.559488
max         2.990384
dtype: float64


In [67]:
noise_loss_df = pd.DataFrame({
    "Train_ID": train_ids_original,
    "Clean_Label": train_df["clean_label"].to_numpy(),
    "Noisy_Label": train_df["noisy_label"].to_numpy(),
    "is_noisy": train_df["is_noisy"].to_numpy(),
    "Training_Loss": training_losses
})

print(noise_loss_df.head())
print(
    noise_loss_df.groupby("is_noisy")[
        "Training_Loss"
    ].describe()
)

   Train_ID  Clean_Label  Noisy_Label  is_noisy  Training_Loss
0      5018            1            1         0       0.324082
1     10999            1            0         1       1.027546
2      3146            1            0         1       1.837602
3      9147            0            0         0       0.262875
4     12955            1            1         0       0.520085
           count      mean       std       min       25%       50%       75%  \
is_noisy                                                                       
0         6400.0  0.291851  0.193044  0.023511  0.184388  0.243852  0.330548   
1         1600.0  1.461925  0.404876  0.121058  1.243351  1.474504  1.709194   

               max  
is_noisy            
0         2.417595  
1         2.990384  


In [68]:
noise_loss_df.to_csv(
    "Noise_GroundTruth_and_TrainingLoss.csv",
    index=False
)

In [69]:
train_df.to_csv(
    "NoisyLabel_TrainingData.csv",
    index=False
)

test_df.to_csv(
    "Clean_TestData.csv",
    index=False
)

# Influence Estimation Area

**Again, you could use other influence analysis methods rather than IF/TC. You can also use any other Influence Function or TracIn implementation. Just Remember: 1. Make sure the package is unform throughout the framework. 2. Generate a Ranked influence list for each Influence Function and TracIn; Only the ranked influence list could be fed into the following analysis code.**

**The default code now use the model list, train set and test set to estimate the influence, and produce a ranked influence list for both IF and TC. The results are then saved in the root directory.**

**Input**:Model list from Training section, Train and Test Set from Data Construction Section   
**Output**: Two ranked Influence Lists for IF and TC.  
**Guideline**: Input -> Influence Estimation Methods -> Influence Matrix -> Output

1. Influence Function: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use influence_matrix.

In [70]:
train_ids = []
test_ids = []
train_samples_np = np.array([x.numpy() for x, y in train_ds])
train_ids = [round(sample[-1] * 1e10) for sample in train_samples_np]

In [71]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

influence_model = model_list[-1]
ihvp_calculator = ExactIHVP(influence_model, train_ds.batch(16))
influence_calculator = FirstOrderInfluenceCalculator(influence_model, train_ds, ihvp_calculator)

influence_matrix = np.zeros((num_test_samples, num_train_samples))

samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            influence_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(influence_matrix, axis=0).reshape(1, -1)
df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(df)

      Train_ID     Score
0         5018  0.191415
1        10999 -0.157258
2         3146 -1.040044
3         9147  0.102674
4        12955  0.186383
...        ...       ...
7995     13382  0.172427
7996     13339  0.125705
7997     15658 -0.200291
7998     12979  0.140438
7999      9910  0.131435

[8000 rows x 2 columns]


2. TracIn: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use TracIn_matrix.

In [72]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

TracIn_matrix = np.zeros((num_test_samples, num_train_samples))
influence_calculator = TracIn(
    model_list, 0.001
)
samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            TracIn_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(TracIn_matrix, axis=0).reshape(1, -1)
TracIn_df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(TracIn_df)

      Train_ID     Score
0         5018  0.030606
1        10999 -0.050818
2         3146 -0.115014
3         9147 -0.000060
4        12955  0.024627
...        ...       ...
7995     13382 -0.000220
7996     13339  0.004371
7997     15658 -0.016247
7998     12979  0.000590
7999      9910  0.040457

[8000 rows x 2 columns]


3. Here we turn both influence lists to the ranked influence lists and then store them for further processing.

In [73]:
df_sorted = df.sort_values(by="Score", ascending=False).reset_index(drop=True)

TracIn_sorted = TracIn_df.sort_values(by="Score", ascending=False).reset_index(drop=True)

In [74]:
TracIn_sorted.to_csv("NoisyLabel_TracIn_Scores.csv",index = False)
df_sorted.to_csv("NoisyLabel_FOIF_Scores.csv",index = False)